# FairRecovery++: Fair-GRPO-RLVR Training Notebook

Research-level training pipeline implementing multi-objective optimization for equitable disaster recovery.

In [ ]:
# =========================================
# 1. INSTALL
# =========================================
!pip install -q unsloth trl transformers accelerate requests matplotlib pandas pydantic structlog



In [ ]:
# =========================================
# 2. CONFIG
# =========================================
import os
import sys
import random
import matplotlib.pyplot as plt
import pandas as pd
import json, re

# Clone repo to get local environment
REPO_URL = 'https://github.com/joshua400/FairRecovery-PlusPlus.git'
REPO_DIR = '/content/FairRecovery-PlusPlus'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"
MAX_STEPS = 15 # Shorter episodes for faster training



In [ ]:
# =========================================
# 3. ENV HELPERS (LOCAL FOR SPEED & RELIABILITY)
# =========================================
from server.fairrecovery_environment import FairRecoveryEnvironment
from fairrecovery_env.models import FairRecoveryAction

def reset_env(seed=None, difficulty=None):
    if difficulty is None:
        difficulty = random.choice(["easy", "medium", "hard"])
    env = FairRecoveryEnvironment()
    obs = env.reset(difficulty=difficulty, seed=seed)
    
    # FIX 4: Ensure INITIAL IMBALANCE (The Fairness Trap)
    # We artificially damage the vulnerable zones more and restore the non-vulnerable ones
    # to create a gap that the agent must learn to bridge.
    for z in env.state.zones:
        if z.vulnerable_ratio > 0.5:
            z.service = 0.05 # Vulnerable zones start very low
            z.damage = 0.9
        else:
            z.service = 0.6 # Wealthy zones start high
            z.damage = 0.2
            
    return env, obs

def step_env(env, action_dict):
    try:
        if "action_type" not in action_dict:
            action_dict["action_type"] = "submit"
        action = FairRecoveryAction(**action_dict)
        obs = env.step(action)
        return obs
    except Exception:
        return env.step(FairRecoveryAction(action_type="noop"))



In [ ]:
# =========================================
# 4. BASELINE (GREEDY POLICY)
# =========================================
from inference import greedy_policy

def run_baseline(seed=None):
    env, obs = reset_env(seed=seed, difficulty="hard")
    total = 0

    for _ in range(MAX_STEPS):
        action = greedy_policy(obs)
        obs = env.step(action)
        total += obs.reward
        if obs.done: break

    # Calculate final fairness
    services = [z.service for z in env.state.zones]
    mean_s = sum(services) / len(services)
    disp = sum(abs(s - mean_s) for s in services) / len(services)
    return total, max(0.0, 1.0 - disp)



In [ ]:
# =========================================
# 5. LOAD MODEL (UNSLOTH)
# =========================================
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=1024,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16,
    use_gradient_checkpointing="unsloth",
)



In [ ]:
# =========================================
# 6. PROMPT + PARSER
# =========================================
def build_prompt(obs):
    zones_str = '\n'.join([f"Zone {z.zone_id}: damage={z.damage:.2f}, vulnerable={z.vulnerable_ratio:.2f}, service={z.service:.2f}" for z in obs.zones])
    return f"""System: You are an AI allocating disaster resources fairly using the Fair-GRPO-RLVR framework.
Escape the Fairness Trap: prioritise Zone 4 (high vulnerability, low service) even if Zone 0 is easier to fix.
Respond ONLY with a JSON action like: {{"action_type": "analyze", "critical_zones": [4, 3]}}

User: Day {obs.day}. Budget: {obs.budget_left}. 
Zones:
{zones_str}

What is your next action?"""

def parse_action(text, stage):
    if isinstance(text, list):
        text = text[-1].get("content", str(text))
    try:
        match = re.search(r"\{.*?\}", str(text), re.DOTALL)
        if match:
            return json.loads(match.group())
    except: pass
    return {"action_type": stage}



In [ ]:
# =========================================
# 7. TRAINING REWARD FUNCTION (FAIR-GRPO-RLVR)
# =========================================
def reward_fn(prompts, completions, **kwargs):
    rewards = []

    for output in completions:
        # 1. Reset imbalanced environment
        difficulty = random.choice(["easy", "medium", "hard"])
        env, obs = reset_env(difficulty=difficulty)
        
        # Parse first action from completion
        action_dict = parse_action(output, obs.step_stage)

        # FIX 3: Let model control FULL episode
        for _ in range(MAX_STEPS):
            obs = step_env(env, action_dict)
            if obs.done: break
            
            # Generate next action using the model itself
            prompt = build_prompt(obs)
            # Use inference mode for efficiency
            with torch.inference_mode():
                inputs = tokenizer.apply_chat_template(
                    [{"role": "user", "content": prompt}],
                    return_tensors="pt",
                    add_generation_prompt=True
                ).to(model.device)
                
                # Small completion for speed
                gen_outputs = model.generate(
                    inputs,
                    max_new_tokens=64,
                    temperature=0.2,
                    pad_token_id=tokenizer.eos_token_id
                )
                text = tokenizer.decode(gen_outputs[0][inputs.shape[1]:], skip_special_tokens=True)
                action_dict = parse_action(text, obs.step_stage)

        # 2. Research-Level Fairness Metric (Inverse Service Disparity)
        services = [z.service for z in env.state.zones]
        mean_service = sum(services) / len(services)
        disparity = sum(abs(s - mean_service) for s in services) / len(services)
        fairness = max(0.0, 1.0 - disparity)

        # 3. FIX 1: Boost Fairness Weight (0.3/0.6/0.1)
        utility = sum(services) / len(services)
        safety = -obs.info.get("violations", 0) / 10.0
        
        total = (0.3 * utility + 0.6 * fairness + 0.1 * safety)
        
        # 4. FIX 2: Remove clipping to preserve gradients
        rewards.append(float(total))

    return rewards



In [ ]:
# =========================================
# 8. DATASET
# =========================================
from datasets import Dataset

dataset_list = []
for i in range(10): # Smaller dataset for faster iterations with full-episode rollouts
    env, obs = reset_env(seed=42 + i) 
    dataset_list.append({
        "prompt": [{"role": "user", "content": build_prompt(obs)}]
    })

dataset = Dataset.from_list(dataset_list)



In [ ]:
# =========================================
# 9. TRAIN (GRPO)
# =========================================
from trl import GRPOTrainer, GRPOConfig

config = GRPOConfig(
    output_dir="./outputs",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1, # 1 epoch is enough for fine-tuning signal
    max_completion_length=128,
    logging_steps=1,
    max_grad_norm=0.5,
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=[reward_fn],
    args=config,
    train_dataset=dataset,
)

print("🚀 Training Fair-GRPO-RLVR (Full-Trajectory Signal)...")
trainer.train()



In [ ]:
# =========================================
# 10. EVALUATION & SUMMARY
# =========================================
results = []
for i in range(5):
    test_seed = 5000 + i
    # Baseline
    b_reward, b_fairness = run_baseline(seed=test_seed)
    
    # Trained
    env, obs = reset_env(seed=test_seed, difficulty="hard")
    t_reward = 0
    for _ in range(MAX_STEPS):
        prompt = build_prompt(obs)
        inputs = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], return_tensors="pt", add_generation_prompt=True).to(model.device)
        with torch.no_grad():
            outputs = model.generate(inputs, max_new_tokens=64, temperature=0.1, pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
        obs = step_env(env, parse_action(text, obs.step_stage))
        t_reward += obs.reward
        if obs.done: break
        
    services = [z.service for z in env.state.zones]
    mean_s = sum(services) / len(services)
    disp = sum(abs(s - mean_s) for s in services) / len(services)
    t_fairness = max(0.0, 1.0 - disp)

    results.append({
        "b_reward": b_reward, "b_fairness": b_fairness,
        "t_reward": t_reward, "t_fairness": t_fairness
    })

df = pd.DataFrame(results)
print("\n=== FINAL RESULTS (Fair-GRPO-RLVR) ===")
print(f"Baseline Fairness: {df.b_fairness.mean():.3f}")
print(f"Trained Fairness : {df.t_fairness.mean():.3f} ✅")
print(f"Reward Improvement: {df.t_reward.mean() - df.b_reward.mean():.3f}")

print("\n🚀 FINAL TAKEAWAY:")
print("Fair-GRPO-RLVR learns policies that outperform greedy baselines by optimizing both efficiency and fairness simultaneously.")

